In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import subprocess

from utils import compose_windows

from sklearn.base import clone
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc, accuracy_score

## Logistic Regression regularization trajecotries:

Here we only investigate what happens to the model weights as we turn up the L2. L1 and ElasticNet are ignored only for simplicity and no other reason.

For each tissue, a DataFrame is created that contains:
1. Metrics (mean and std of ACC and AUC ROC) computed on **CROSS-VALIDATION**
2. Lists of most important motifs based on weights of the model **trained on all of the data**

## TODO

Zrobić to w statsmodels, tak jak opisano na Mattermot. W statsmodels trzeba ręczenie dodać wyraz wolny w modelu liniowym.

Dodać do df informacje o wagach i zapisać to w postaci long

Narysować współczynninki z regresji logistycznej z przedziałami ufności

Sprawdzić czemu w obecnych wynikach wszystkie wiersze wychodzą identycznie

In [16]:
windows=["06-08", "10-12", "14-16"]
tissues=["Neuroblasts", "Neurons", "Glia"]
model_name = "Logistic Regression"
model_short = "LR"

n_splits = 10   # cross-validation splits

lambda_range = np.linspace(0.0, 4, 21)      # 0.0, 0.2, ... , 3.8, 4.0

In [ ]:
def reg_trajectory(tissue: str, n_splits: int = 10, top_n: int = 20, lambda_range: np.linspace = np.linspace(0,1, 11), motif_annotations_path: str = "data/motif_names.tsv") -> pd.DataFrame :

    # Load windows
    X, y, composite = compose_windows(tissue)

    results = []

    for l2_value in lambda_range:

        # -----------------------
        # Cross-validation
        # -----------------------
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)
        probs, trues, accs = [], [], []

        base_classifier = LogisticRegression(
            penalty="l2",
            l1_ratio=l2_value,
            n_jobs=-1
        )

        for i, (train_idx, test_idx) in enumerate(skf.split(X, composite)):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            classifier = clone(base_classifier)
            classifier.fit(X_train, y_train)

            p = classifier.predict_proba(X_test)[:, 1]
            probs.append(p)
            trues.append(y_test.values)
            accs.append(accuracy_score(y_test, (p > 0.5).astype(int)))

        # Accuracy metrics
        mean_acc = np.mean(accs)
        std_acc = np.std(accs)

        # ROC AUC metrics
        roc_aucs = []
        for true, prob in zip(trues, probs):
            fpr, tpr, _ = roc_curve(true, prob)
            roc_aucs.append(auc(fpr, tpr))

        mean_auc = np.mean(roc_aucs)
        std_auc = np.std(roc_aucs)

        # Load annotations, then train on full dataset

        annot = pd.read_csv(motif_annotations_path, sep="\t")

        num_mismatches = len(X.columns) - (X.columns == annot["id"]).sum()
        assert num_mismatches == 0, f"Invalid annotations! There are {num_mismatches} mismatches in motif codes!"

        new_columns = annot["name"].astype(str) #+ "  -  (" + annot["id"].astype(str) + ")"
        X.columns = new_columns

        # train on full dataset
        full_classifier = clone(base_classifier)
        full_classifier.fit(X, y)

        weights = full_classifier.coef_.ravel()

        coef_df = pd.DataFrame({
            "feature": X.columns,
            "weight": weights
        })

        coef_df["abs_weight"] = coef_df["weight"].abs()
        coef_df = coef_df.sort_values("abs_weight", ascending=False)

        top_df = coef_df.head(top_n)

        top_motifs = top_df["feature"].tolist()
        mean_abs_top_weights = top_df["abs_weight"].mean()

        # Store results
        row = {
            "mean abs(top weights)": mean_abs_top_weights,
            "mean AUC ROC": mean_auc,
            "std AUC ROC": std_auc,
            "mean Accuracy": mean_acc,
            "std Accuracy": std_acc
        }

        # Add motif columns: motif 1, motif 2, ...
        for i, motif in enumerate(top_motifs, start=1):
            row[f"motif {i}"] = motif

        results.append(row)

    # Construct final dataframe
    trajectory = pd.DataFrame(results, index=lambda_range)
    trajectory.index.name = "l2_ratio"

    trajectory.to_csv(f"results/data/regularization_trajectories/{tissue}_top_motifs.tsv", sep="\t")

    return trajectory


In [42]:
def reg_trajectory(tissue: str, n_splits: int = 10, top_n: int = 20, lambda_range: np.linspace = np.linspace(0,1, 11), motif_annotations_path: str = "data/motif_names.tsv") -> pd.DataFrame :

    # Load windows
    X, y, composite = compose_windows(tissue)

    results = []

    for l2_value in lambda_range:

        # -----------------------
        # Cross-validation
        # -----------------------
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)
        probs, trues, accs = [], [], []

        base_classifier = LogisticRegression(
            penalty="l2",
            l1_ratio=(1-l2_value),
            n_jobs=-1
        )

        for i, (train_idx, test_idx) in enumerate(skf.split(X, composite)):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            classifier = clone(base_classifier)
            classifier.fit(X_train, y_train)

            p = classifier.predict_proba(X_test)[:, 1]
            probs.append(p)
            trues.append(y_test.values)
            accs.append(accuracy_score(y_test, (p > 0.5).astype(int)))

        # Accuracy metrics
        mean_acc = np.mean(accs)
        std_acc = np.std(accs)

        # ROC AUC metrics
        roc_aucs = []
        for true, prob in zip(trues, probs):
            fpr, tpr, _ = roc_curve(true, prob)
            roc_aucs.append(auc(fpr, tpr))

        mean_auc = np.mean(roc_aucs)
        std_auc = np.std(roc_aucs)

        # Load annotations, then train on full dataset

        annot = pd.read_csv(motif_annotations_path, sep="\t")

        # Build mapping: motif code -> motif name
        code_to_name = (
            annot
            .drop_duplicates("id")
            .set_index("id")["name"]
            .astype(str)
            .to_dict()
        )

        # Map columns; keep original code if annotation is missing
        mapped_columns = (
            X.columns
            .to_series()
            .map(code_to_name)
            .fillna(pd.Series(X.columns, index=X.columns))
        )
        X = X.copy()
        X.columns = mapped_columns

        # train on full dataset
        full_classifier = clone(base_classifier)
        full_classifier.fit(X, y)

        weights = full_classifier.coef_.ravel()

        coef_df = pd.DataFrame({
            "feature": X.columns,
            "weight": weights
        })

        coef_df["abs_weight"] = coef_df["weight"].abs()
        coef_df = coef_df.sort_values("abs_weight", ascending=False)

        top_df = coef_df.head(top_n)

        top_motifs = top_df["feature"].tolist()
        mean_abs_top_weights = top_df["abs_weight"].mean()

        # Store results
        row = {
            "mean abs(top weights)": mean_abs_top_weights,
            "mean AUC ROC": mean_auc,
            "std AUC ROC": std_auc,
            "mean Accuracy": mean_acc,
            "std Accuracy": std_acc
        }

        # Add motif columns: motif 1, motif 2, ...
        for i, motif in enumerate(top_motifs, start=1):
            row[f"motif {i}"] = motif

        results.append(row)

    # Construct final dataframe
    trajectory = pd.DataFrame(results, index=lambda_range)
    trajectory.index.name = "l2_ratio"
    os.makedirs("results/data/regularization_trajectories", exist_ok=True)
    trajectory.to_csv(f"results/data/regularization_trajectories/{tissue}_top_motifs.tsv", sep="\t")

    return trajectory


In [43]:
for t in tissues:
    _ = reg_trajectory(lambda_range=np.linspace(0,1 , 11), tissue=t)

/home/olek/miniconda3/envs/env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/home/olek/miniconda3/envs/env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/home/olek/miniconda3/envs/env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/home/olek/miniconda3/envs/env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/home/olek/miniconda3/envs/env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'el

In [ ]:
# def reg_trajectory(tissue: str, n_splits: int = 10, lambda_range: np.linspace = np.linspace(0,1, 11), motif_annotations_path: str = "data/motif_names.tsv") -> pd.DataFrame :
#     # Load windows
#     X, y, composite = compose_windows(tissue)

#     for l1_value in lambda_range:

#         # Cross-validation
#         skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)
#         probs, trues, accs = [], [], []
#         fold_index = {}

#         classifier = LogisticRegression(penalty="l1" ,l1_ratio=l1_value)

        
#         for (i, (train_idx, test_idx)) in enumerate(skf.split(X, composite)):
#             X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
#             y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
#             fold_index[i] = (train_idx, test_idx)

#             classifier = clone(classifier) # this creates an UNFITTED model with the same parameters as before
#             classifier.fit(X_train, y_train)

#             p = classifier.predict_proba(X_test)[:, 1]
#             probs.append(p)
#             trues.append(y_test.values)
#             accs.append(accuracy_score(y_test, (p > 0.5).astype(int)))

#         # Compute metrics
#         mean_acc = np.mean(accs)
#         std_acc = np.std(accs)

#         fprs, tprs, roc_aucs = [], [], []
#         for (true, prob) in zip(trues, probs):
#             fpr, tpr, _ = roc_curve(true, prob)
#             fprs.append(fpr)
#             tprs.append(tpr)
#             roc_aucs.append(auc(fpr, tpr))

#         mean_auc = np.mean(roc_aucs)
#         std_auc = np.std(roc_aucs)

#         # Teraz klasyfikator na całych danych i wagi z niego

        

#     # konstrukcja tkankowego df poza pętlą
#     trajectory = pd.DataFrame()

#     trajectory.to_csv(f"results/data/regularization_trajectories/{t}_top_motifs.tsv", sep="\t")

#     return trajectory

In [44]:
_

,mean abs(top weights),mean AUC ROC,std AUC ROC,mean Accuracy,std Accuracy,motif 1,motif 2,motif 3,motif 4,motif 5,...,motif 11,motif 12,motif 13,motif 14,motif 15,motif 16,motif 17,motif 18,motif 19,motif 20
l2_ratio,,,,,,,,,,,,,,,,,,,,,
0.0,1.035052,0.713512,0.050482,0.706667,0.029533,SU(HW),CTCF,Cf2,CG4404,Myb,...,dmrt99B,vvl,lbe,net,peb,lola,Med,sug,pad,CG12605
0.1,1.035052,0.713512,0.050482,0.706667,0.029533,SU(HW),CTCF,Cf2,CG4404,Myb,...,dmrt99B,vvl,lbe,net,peb,lola,Med,sug,pad,CG12605
0.2,1.035052,0.713512,0.050482,0.706667,0.029533,SU(HW),CTCF,Cf2,CG4404,Myb,...,dmrt99B,vvl,lbe,net,peb,lola,Med,sug,pad,CG12605
0.3,1.035052,0.713512,0.050482,0.706667,0.029533,SU(HW),CTCF,Cf2,CG4404,Myb,...,dmrt99B,vvl,lbe,net,peb,lola,Med,sug,pad,CG12605
0.4,1.035052,0.713512,0.050482,0.706667,0.029533,SU(HW),CTCF,Cf2,CG4404,Myb,...,dmrt99B,vvl,lbe,net,peb,lola,Med,sug,pad,CG12605
0.5,1.035052,0.713512,0.050482,0.706667,0.029533,SU(HW),CTCF,Cf2,CG4404,Myb,...,dmrt99B,vvl,lbe,net,peb,lola,Med,sug,pad,CG12605
0.6,1.035052,0.713512,0.050482,0.706667,0.029533,SU(HW),CTCF,Cf2,CG4404,Myb,...,dmrt99B,vvl,lbe,net,peb,lola,Med,sug,pad,CG12605
0.7,1.035052,0.713512,0.050482,0.706667,0.029533,SU(HW),CTCF,Cf2,CG4404,Myb,...,dmrt99B,vvl,lbe,net,peb,lola,Med,sug,pad,CG12605
0.8,1.035052,0.713512,0.050482,0.706667,0.029533,SU(HW),CTCF,Cf2,CG4404,Myb,...,dmrt99B,vvl,lbe,net,peb,lola,Med,sug,pad,CG12605
